# BaoStockDB

`BaoStockDB` 是基于 [BaoStock](http://baostock.com/baostock/) 在线 API 构建的因子库，主要用于**演示和测试**。通过 HTTP 请求实时获取数据，无需本地数据库。

> 注意：BaoStockDB 仅实现了 BaoStock API 的部分接口，不适合生产环境使用。可用于快速验证因子逻辑或编写示例代码。

## 连接参数

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `UserID` | `"anonymous"` | BaoStock 登录用户 ID |
| `Pwd` | `"123456"` | BaoStock 登录密码 |

通常无需配置文件，使用默认匿名账号即可。如需自定义，配置文件为 `~/QuantStudioConfig/BaoStockDBConfig.json`。

## 因子表类型

BaoStockDB 支持 6 种因子表类型，对应 BaoStock 的不同 API 模式：

| 类型 | 参数模式 | 说明 |
|------|----------|------|
| `DTRangeTable` | `start_date` + `end_date` | 时间区间查询，如 K 线、业绩快报 |
| `DTTable` | `date` | 逐时点查询，如行业分类、成分股、复权因子 |
| `QuarterTable` | `year` + `quarter` | 按季度查询，如季频财务指标 |
| `YearTable` | `year` | 按年份查询，如除权除息信息 |
| `MacroDataTable` | `start_date` + `end_date` | 宏观经济数据，无证券代码维度 |
| `StockBasicTable` | `code` | 证券基本资料，无日期参数 |

## 全部因子表（24 张）

| 表名 | 类型 | 说明 |
|------|------|------|
| A股K线数据 | DTRangeTable | 日/周/月 K 线，含估值指标 |
| 季度业绩快报 | DTRangeTable | 季度业绩快报数据 |
| 季度业绩预告 | DTRangeTable | 季度业绩预告数据 |
| 行业分类 | DTTable | 证监会行业分类 |
| 每日A股K线 | DTTable | 单日 A 股 K 线查询 |
| 每日ETF K线 | DTTable | 单日 ETF K 线查询 |
| 每日复权因子 | DTTable | 前/后复权因子 |
| 每日全部A股 | DTTable | 某日全部在市A股列表 |
| 上证50成分股 | DTTable | 上证 50 成分股列表 |
| 沪深300成分股 | DTTable | 沪深 300 成分股列表 |
| 中证500成分股 | DTTable | 中证 500 成分股列表 |
| 季度盈利能力 | QuarterTable | ROE、净利率、毛利率等 |
| 季度营运能力 | QuarterTable | 应收/存货/资产周转率等 |
| 季度成长能力 | QuarterTable | 净利润/净资产/每股收益同比增长 |
| 季度偿债能力 | QuarterTable | 流动/速动比率、资产负债率等 |
| 季度现金流量 | QuarterTable | 经营现金流相关指标 |
| 季度杜邦指数 | QuarterTable | 杜邦分析分解指标 |
| 除权除息信息 | YearTable | 分红送股除权除息明细 |
| 存款利率 | MacroDataTable | 央行存款基准利率 |
| 贷款利率 | MacroDataTable | 央行贷款基准利率 |
| 存款准备金率 | MacroDataTable | 存款准备金率调整 |
| 货币供应量 | MacroDataTable | M0/M1/M2 月度数据 |
| 货币供应量(年底余额) | MacroDataTable | M0/M1/M2 年底余额 |
| 证券基本资料 | StockBasicTable | 股票代码/名称/上市日期/状态 |

## 使用步骤

先 `connect()`，用 `TableNames`、`FactorNames` 确认当前可用接口，再选择日期和证券调用 `readData()`。返回轴顺序为“因子 × 时点 × 证券”；财务与宏观接口的查询频率不同，不能将缺失值直接理解为零。下面的读取示例需要联网，服务可用性与数据范围由 BaoStock 决定。

In [ ]:
import logging
import warnings
from QuantStudio.Core import setDefaultLogLevel
warnings.filterwarnings("ignore")
setDefaultLogLevel(logging.WARNING)

In [ ]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

In [ ]:
# 创建因子库对象并 connect（需要联网）
from QuantStudio.Factor.BaoStockDB import BaoStockDB

FDB = BaoStockDB().connect()

In [ ]:
print(qs_help(FDB))

In [ ]:
# 可用的因子表（24 张）
print(FDB.TableNames[:5])

# 时点与 ID 获取

BaoStockDB 提供了基本的交易日和股票代码查询。

In [ ]:
# 获取交易日序列
DTs = FDB.getTradeDay(start_date=dt.datetime(2022, 1, 1), end_date=dt.datetime(2022, 1, 20))
print(DTs[:5])

In [ ]:
# 获取当前在市的全体 A 股
IDs = FDB.getStockID()
print(IDs[:5])

# DTRangeTable — A股K线数据

`DTRangeTable` 按时间区间查询，一次 API 请求获取整个日期范围的数据。适用于 K 线、业绩快报、业绩预告等。

核心参数：
- `LookBack`：缺失填充回溯天数（默认 0）
- `APIArgs`：透传给 BaoStock API 的额外参数（如 `frequency`、`adjustflag`）

In [ ]:
# 获取因子表 — A股K线数据（DTRangeTable）
FT = FDB.getTable("A股K线数据", args={"LookBack": 0})

In [ ]:
print(qs_help(FT))

In [ ]:
# 因子列表
print(FT.FactorNames)

In [ ]:
# 读取因子表数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = FT.readData(factor_names=["open", "close"], ids=IDs, dts=DTs)
print("因子表数据 (Panel):")
print(Data)

In [ ]:
# DTRangeTable 无 Field 参数示例：季度业绩快报
# 与 A股K线数据 不同，此表的 ArgInfo 中没有 Field 类型参数，不传 fields 给 API
FT = FDB.getTable("季度业绩快报")
print(f"因子: {FT.FactorNames}")

DTs = [dt.datetime(2024, 6, 30)]
Data = FT.readData(factor_names=["performanceExpressTotalAsset", "performanceExpressROEWa"], ids=["600000.SH"], dts=DTs)
print(Data)

# DTTable — 行业分类 / 成分股 / 复权因子

`DTTable` 逐时点查询，每个日期发一次 API 请求。适用于行业分类、成分股、复权因子、单日 K 线等。

In [ ]:
# DTTable 示例：行业分类
FT = FDB.getTable("行业分类")

DTs = [dt.datetime(2024, 6, 28)]
Data = FT.readData(factor_names=["industry", "industryClassification"], ids=["600000.SH", "000001.SZ"], dts=DTs)
print(Data)

# QuarterTable

In [ ]:
# QuarterTable 示例：季度盈利能力
FT = FDB.getTable("季度盈利能力")

DTs = [dt.datetime(2024, 3, 31), dt.datetime(2024, 6, 30), dt.datetime(2024, 9, 30)]
Data = FT.readData(factor_names=["roeAvg", "npMargin", "epsTTM"], ids=["600000.SH"], dts=DTs)
print(Data)

# YearTable

In [ ]:
# YearTable 示例：除权除息信息
FT = FDB.getTable("除权除息信息")
print(f"表类型: {type(FT).__name__}, 因子: {FT.FactorNames}")

DTs = [dt.datetime(2024, 6, 30)]
Data = FT.readData(factor_names=["dividCashPsBeforeTax", "dividStocksPs"], ids=["600000.SH"], dts=DTs)
print(Data)

# MacroDataTable

In [ ]:
# MacroDataTable 示例：存款利率
# MacroDataTable 无证券代码维度，ids 传入占位即可
FT = FDB.getTable("存款利率")

DTs = [dt.datetime(2015, 3, 1), dt.datetime(2015, 10, 24)]
Data = FT.readData(factor_names=["demandDepositRate", "fixedDepositRate1Year"], ids=["ALL"], dts=DTs)
print(Data)

In [ ]:
# MacroDataTable 特殊情况：货币供应量（无 Date 字段）
# 当 MacroDataTable 的 FactorInfo 中没有 Date 类型字段时，使用 statYear + statMonth 构造日期
FT = FDB.getTable("货币供应量")
print(f"表类型: {type(FT).__name__}, 因子: {FT.FactorNames}")

DTs = [dt.datetime(2010, 1, 31), dt.datetime(2010, 3, 31)]
Data = FT.readData(factor_names=["m0Month", "m1Month", "m2Month"], ids=["ALL"], dts=DTs)
print(Data.iloc[:, :, 0])

In [ ]:
# MacroDataTable 特殊情况：货币供应量(年底余额)（无 Date 字段，且无 statMonth 字段）
# 当 MacroDataTable 的 FactorInfo 中没有 Date 类型字段时，使用 statYear 构造日期
FT = FDB.getTable("货币供应量(年底余额)")
print(f"表类型: {type(FT).__name__}, 因子: {FT.FactorNames}")

DTs = [dt.datetime(2010, 12, 31), dt.datetime(2011, 12, 31)]
Data = FT.readData(factor_names=["m0Year", "m0YearYOY", "m1Year"], ids=["ALL"], dts=DTs)
print(Data.iloc[:, :, 0])

# StockBasicTable

In [ ]:
# StockBasicTable 示例：证券基本资料
# StockBasicTable 无日期参数，dts 传入占位即可
FT = FDB.getTable("证券基本资料")
print(f"表类型: {type(FT).__name__}, 因子: {FT.FactorNames}")

Data = FT.readData(factor_names=["code_name", "ipoDate", "status"], ids=["600000.SH", "000001.SZ"], dts=[dt.datetime(2024, 12, 31)])
print(Data)

# 因子

通过 `FT.getFactor()` 获取的因子对象与其他因子库中的因子完全一致，同样支持运算符重载和衍生因子。详见 **[基本框架](基本框架.ipynb)** 和 **[因子开发](因子开发.ipynb)**。